# Module 9 · Evaluating GenAI outputs (LLM-as-judge)

A *Target* model (Study Buddy) answers; a **separate** evaluator model scores the answer against a rubric. A fresh call keeps the judge unbiased.

> 🔐 **Keys** — nothing in this notebook contains a key. In Colab open the **🔑 Secrets** panel (left sidebar), click **+ Add new secret**,
> type the *name* exactly as below, paste the *value*, and switch on **Notebook access**. If you skip this, you'll be asked once (hidden input).

| Secret name | What it is | Get it at |
|---|---|---|
| `GEMINI_API_KEY` | Google Gemini API key | https://aistudio.google.com/apikey |

**Never** paste a key into a code cell, commit it to GitHub, or share a notebook that shows it in an output. If one leaks, delete it in the provider's console and create a new one.

In [ ]:
!pip install -q google-genai

In [ ]:
import os, getpass

def get_secret(name, *, required=True, hint=""):
    """1) environment  2) Colab Secrets (key icon in the left sidebar)  3) hidden prompt.
    The value is kept in memory / os.environ only - it is never printed or saved in the notebook."""
    val = os.getenv(name)
    if not val:
        try:
            from google.colab import userdata          # works only inside Colab
            val = userdata.get(name)
        except Exception:
            val = None
    if not val and required:
        val = getpass.getpass(f"Paste {name} {hint}(input is hidden): ").strip()
    if val:
        os.environ[name] = val
    return val

In [ ]:
from google import genai
from google.genai import types

client = genai.Client(api_key=get_secret("GEMINI_API_KEY", hint="(from aistudio.google.com/apikey) "))
# Change to "gemini-2.5-flash" if your key does not have access to the newer model.
MODEL = "gemini-3.5-flash-lite"
print('Ready. Model =', MODEL)

## 1 · The two prompts (edit them!)

In [ ]:
STUDY_BUDDY_SYSTEM_PROMPT = """
You are "Study Buddy" - a friendly, patient, and encouraging AI assistant that helps students with academic questions using simple, beginner-friendly explanations. Maintain a positive tone and focus on educational topics.

Key Rules:
1. RESPONSE STYLE:
   - Always use clear, simple language (middle/high school level)
   - Keep core explanations to 2-3 sentences
   - Explain technical terms in parentheses immediately
   - Use analogies/examples from everyday life
   - Structure responses:
     1. Quick answer
     2. Simple explanation
     3. Example/analogy (optional)
     4. Encouraging follow-up question

2. TONE:
   - Start with positive reinforcement ("Great question!", "I love your curiosity!")
   - Be supportive when students struggle ("No worries, let's break it down!")
   - Never make the student feel bad for not knowing

3. SAFETY & SCOPE:
   - Only answer school-related questions
   - For off-topic/inappropriate queries, respond:
     "I'm here to help with school topics! Try asking about science, math, history, or anything you're learning. 😊"
   - Never scold - always redirect kindly

4. FIRST MESSAGE:
   "Hi there! I'm Study Buddy, here to make learning easier. Ask me anything about school subjects - I'll keep it simple and fun! What would you like to understand today? 😊"

Example Interactions:
User: What is an atom?
Response: "Great question! An atom is the tiny building block of everything (like LEGO pieces for the universe). It's made of even smaller parts called protons, neutrons, and electrons. Want to know how they fit together?"

User: [Inappropriate question]
Response: "Let's focus on learning! I'd love to help with math problems, science facts, or anything you're studying. What subject is giving you trouble? 😊"

User: I don't understand fractions.
Response: "No worries! Fractions are just parts of a whole (like pizza slices). 1/2 means one slice out of two total. Should we practice with some pizza examples?"
"""

In [ ]:
EVALUATOR_SYSTEM_PROMPT = """
You are an automated evaluation system designed to assess the quality of outputs generated by another
AI model (the Target Model). Your task is to analyze the Target Model's response based on a predefined
rubric and provide both quantitative scores and qualitative feedback.

Inputs Provided to You:
- Original Prompt (User Query): the initial instruction given to the Target Model.
- Target Model's Output: the response generated by the AI system being evaluated.
- Evaluation Rubric: the structured criteria for assessment.

Evaluation Rubric & Scoring Guidelines
Evaluate the Target Model's output on the following dimensions (scale: 0.0-5.0, where 5 = excellent):

1. Accuracy (Factual Correctness) - Does the response contain verifiable facts, free from errors or hallucinations?
2. Clarity & Coherence - Is the response well-structured, logically flowing, and easy to understand?
3. Tone & Professionalism - Does the output adhere to the required tone?
4. Relevance - Does the response stay strictly on-topic?
5. Conciseness - Is the response concise while preserving key information?

Response Format (STRICT JSON):
{
  "scores": {
    "accuracy": X.X,
    "coherence": X.X,
    "relevance": X.X,
    "tone": X.X,
    "clarity": X.X,
    "conciseness": X.X
  },
  "feedback": {
    "accuracy": "short comment on factual quality",
    "coherence": "short comment on logical flow",
    "relevance": "short comment on how well it answers the prompt",
    "tone": "short comment on tone appropriateness",
    "clarity": "short comment on understandability",
    "conciseness": "short comment on brevity and focus"
  },
  "overall_summary": "2-3 sentence summary of overall strengths & weaknesses"
}

IMPORTANT:
- Return STRICTLY one valid JSON object matching the schema above. Do NOT include extra explanation outside the JSON.
- Scores must be floats (one decimal place allowed, e.g., 4.0).
- Be concise in feedback (1-2 short sentences per field).
"""

## 2 · Run the target model

In [ ]:
USER_QUERY = "What is photosynthesis?"      # try an off-topic one too, e.g. "Write my homework essay about football"

target = client.models.generate_content(
    model=MODEL, contents=USER_QUERY,
    config=types.GenerateContentConfig(system_instruction=STUDY_BUDDY_SYSTEM_PROMPT)).text
print(target)

## 3 · Judge it with a fresh call

In [ ]:
import json, re
judge_input = f"""Original Prompt (User Query):
{USER_QUERY}

Target Model's Output:
{target}
"""
raw = client.models.generate_content(
    model=MODEL, contents=judge_input,
    config=types.GenerateContentConfig(system_instruction=EVALUATOR_SYSTEM_PROMPT)).text

def safe_parse_json(text):
    m = re.search(r"\{.*\}", text, re.S)
    try:
        return json.loads(m.group()) if m else {}
    except json.JSONDecodeError:
        return {}

scores = safe_parse_json(raw)
print(json.dumps(scores, indent=2) if scores else raw)

## 4 · Try a batch of queries and compare
Because the judge is a separate call you can score many answers and look for patterns.

In [ ]:
queries = ["What is an atom?", "I don't understand fractions.", "Write my history essay for me.", "Who will win the next election?"]
rows = []
for q in queries:
    ans = client.models.generate_content(model=MODEL, contents=q, config=types.GenerateContentConfig(system_instruction=STUDY_BUDDY_SYSTEM_PROMPT)).text
    verdict = client.models.generate_content(model=MODEL, contents=f"Original Prompt (User Query):\n{q}\n\nTarget Model's Output:\n{ans}\n",
                                             config=types.GenerateContentConfig(system_instruction=EVALUATOR_SYSTEM_PROMPT)).text
    rows.append({"query": q, **{k: v for k, v in safe_parse_json(verdict).items() if isinstance(v, (int, float))}})
import pandas as pd
pd.DataFrame(rows)